### run this line inside the scriptlet section of OpenOnDemand before starting this notebook: 
`export SLURM_NTASKS_PER_NODE="${SLURM_TASKS_PER_NODE:-${SLURM_NTASKS:-1}}"`

This sets it to SLURM_TASKS_PER_NODE if available, else SLURM_NTASKS, else 1.

This is necessary for loading the scvi_model

In [ ]:
samples = ["A1", "A2", "B2", "C2", "D1"]

In [ ]:
import spatialdata as sd
import plotnine as p9
import scvi
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

In [ ]:
# loading zarr
sdata = sd.read_zarr("/scratch/leuven/357/vsc35768/spatial-transcriptomics/intermediate_results/20260119.zarr")
sdata

In [ ]:
# loading the model
scvi_model = scvi.model.SCVI.load("intermediate_results/model_20260119.")
adata_scvi = scvi_model.adata
adata_scvi

## Differential Expression analysis - scVI

In [ ]:
adata_scvi.obs["cell_type"].value_counts()

In [ ]:
# for multiple comparisons
cell_type_1 = "Astrocytes Protoplasmic"
cell_idx1 = adata_scvi.obs["cell_type"] == cell_type_1
print(sum(cell_idx1), "cells of type", cell_type_1)

cell_type_2 = ["DG Granular Neurons", "CA1 Neurons", "CA2/CA3 Neurons", "Microglia"]
cell_idx2 = adata_scvi.obs["cell_type"].isin(cell_type_2)  # Use .isin() for list
print(sum(cell_idx2), "cells of type", cell_type_2)

In [ ]:
# for single comparisons
cell_type_1 = "Astrocytes Protoplasmic"
cell_idx1 = adata_scvi.obs["cell_type"] == cell_type_1
print(sum(cell_idx1), "cells of type", cell_type_1)

cell_type_2 = "DG Granular Neurons"
cell_idx2 = adata_scvi.obs["cell_type"] == cell_type_2
print(sum(cell_idx2), "cells of type", cell_type_2)

In [ ]:
de_change_uniform = scvi_model.differential_expression(
    idx1=cell_idx1,  
    idx2=cell_idx2,
    weights="uniform",
    batch_correction = True,
    filter_outlier_cells = True,
    mode = "change",
)

In [ ]:
de_change_uniform["log10_pscore"] = np.log10(de_change_uniform["proba_not_de"])
de_change_uniform = de_change_uniform.join(adata_scvi.var, how="inner")
de_change_uniform.head(60)

In [ ]:
de_change_importance = scvi_model.differential_expression(
    idx1=cell_idx1, 
    idx2=cell_idx2,
    weights="importance",
    filter_outlier_cells=True,
    batch_correction=True,
    mode="change",
)

In [ ]:
de_change_importance["log10_pscore"] = np.log10(de_change_importance["proba_not_de"])
de_change_importance = de_change_importance.join(adata_scvi.var, how="inner")
de_change_importance.head(60)

In [ ]:
de_comp = pd.concat(
    [
        de_change_importance.assign(flavor="importance"),
        de_change_uniform.assign(flavor="uniform"),
    ]
)

(
    p9.ggplot(de_comp, p9.aes("lfc_mean", "-log10_pscore"))
    + p9.geom_point(
    )  # Plot other genes with transparence
    + p9.labs(x="LFC mean", y="Significance score (higher is more significant)")
    + p9.facet_wrap("flavor")
)

In [ ]:
# Define genes you want to highlight
genes_of_interest = [
    "Gpr37l1", 
    "Vcam1", 
    "Hepacam", 
    "Vcan", 
    "Neo1", 
    "Ephb4", 
    "Ntm", 
    "Alcam", 
    "Cadm4", 
    "Unc13b", 
    "Lsamp"
] 

# Add column for FDR significance
de_comp["is_significant"] = (
    de_change_importance.assign(flavor="importance")["is_de_fdr_0.05"] | 
    de_change_uniform.assign(flavor="uniform")["is_de_fdr_0.05"]
)

# Add highlight column for genes of interest
de_comp["is_gene_of_interest"] = de_comp.index.isin(genes_of_interest)

# Create color categories
de_comp["color_group"] = "Not significant"
de_comp.loc[de_comp["is_significant"], "color_group"] = "Significant (FDR < 0.05)"
de_comp.loc[de_comp["is_gene_of_interest"], "color_group"] = "Gene of interest"

# Plot
(
    p9.ggplot(de_comp, p9.aes("lfc_mean", "-log10_pscore"))
    + p9.geom_point(p9.aes(color="color_group"), alpha=0.6, size=2)
    + p9.scale_color_manual(values={
        "Not significant": "lightgray",
        "Significant (FDR < 0.05)": "steelblue",
        "Gene of interest": "darkblue"
    })
    + p9.geom_text(
        data=de_comp[de_comp["is_gene_of_interest"]], 
        mapping=p9.aes(label=de_comp[de_comp["is_gene_of_interest"]].index),
        nudge_y=0.1,
        size=8
    )
    + p9.labs(x="LFC mean", y="Significance score (higher is more significant)")
    + p9.facet_wrap("flavor")
    + p9.theme_minimal()
)

In [ ]:
# Define genes you want to highlight
genes_of_interest = [
    "Gpr37l1", 
    "Vcam1", 
    "Hepacam", 
    "Plxnb1",
    # "Vcan", 
    # "Neo1", 
    # "Ptprf",
    # "Sirpa"
    # "Adgrl1",
    # "Ephb4", 
    # "Ntm", 
    # "Alcam", 
    # "Cadm4", 
    # "Unc13c", 
    # "Lsamp",
    # "Csf1r",
    # "Sirpa",
    # "C1qa",
    # "Gfap"
] 

# Add column for FDR significance (only for uniform)
de_comp["is_significant"] = de_change_uniform.assign(flavor="importance")["is_de_fdr_0.05"]

# Add highlight column for genes of interest
de_comp["is_gene_of_interest"] = de_comp.index.isin(genes_of_interest)

# Create color categories
de_comp["color_group"] = "Not significant"
de_comp.loc[de_comp["is_significant"], "color_group"] = "Significant (FDR < 0.05)"
de_comp.loc[de_comp["is_gene_of_interest"], "color_group"] = "Gene of interest"

# Filter to only uniform and plot
de_comp_uniform = de_comp[de_comp["flavor"] == "importance"]

(
    p9.ggplot(de_comp_uniform, p9.aes("lfc_mean", "-log10_pscore"))
    + p9.geom_point(p9.aes(color="color_group"), alpha=0.6, size=2)
    + p9.scale_color_manual(values={
        "Not significant": "lightgray",
        "Significant (FDR < 0.05)": "steelblue",
        "Gene of interest": "darkblue"
    })
    + p9.geom_text(
        data=de_comp_uniform[de_comp_uniform["is_gene_of_interest"]], 
        mapping=p9.aes(label=de_comp_uniform[de_comp_uniform["is_gene_of_interest"]].index),
        nudge_y=0.1,
        size=8
    )
    + p9.labs(x="LFC mean", y="Significance score (higher is more significant)")
    + p9.theme_minimal()
)

## Differential expression analysis - pyDESeq2

In [ ]:
adata_scvi

In [ ]:
# build group label
grp = adata_scvi.obs[["sample_id", "cell_type"]].astype(str).agg("__".join, axis=1)
grp.value_counts().shape

In [ ]:
# drop small groups
min_cells_per_group = 10
keep_groups = grp.value_counts()[lambda s: s >= min_cells_per_group].index
mask = grp.isin(keep_groups)

In [ ]:
# subset adata and get count matrix
adata_f = adata_scvi[mask].copy()
X = adata_f.layers["raw_counts"]
X = X.tocsr() if sp.issparse(X) else sp.csr_matrix(X)
grp_f = grp[mask]

In [ ]:
# sum counts per group
groups = pd.Index(grp_f.unique(), name="pb_id")
rows = []

for g in groups:
    idx = np.where(grp_f.values == g)[0]
    r = X[idx].sum(axis=0)            # could be numpy.matrix or sparse, depending on X
    r = sp.csr_matrix(r)              # force 2-D sparse (1 x n_genes)
    rows.append(r)

pb_counts = sp.vstack(rows, format="csr")
print(pb_counts.shape)

In [ ]:
# create pseudobulk metadata
pb_obs = groups.to_series().str.split("__", expand=True)
pb_obs.columns = ["sample_id", "cell_type"]
pb_obs.index = groups

In [ ]:
# build the pseudobulk AnnData object
pb = ad.AnnData(X=pb_counts, obs=pb_obs, var=adata_scvi.var.copy())

In [ ]:
pb.obs

In [ ]:
# add animal id
sample_to_animal = {
    "A1": "animal_1", 
    "A2": "animal_1",
    "B2": "animal_2", 
    "C2": "animal_2",
    "D1": "animal_3",
}
pb.obs["animal_id"] = pb.obs["sample_id"].astype(str).map(sample_to_animal)


In [ ]:
pb.obs

In [ ]:
pd.crosstab(pb.obs["animal_id"], pb.obs["cell_type"])

In [ ]:
pb.shape

In [ ]:
# set cell types to compare
ct1 = "Astrocytes Protoplasmic"
ct2 = "CA2/CA3 Neurons"
#ct2_list = ["CA1 Neurons", "CA2/CA3 Neurons", "DG Granular Neurons"]

In [ ]:
# if multiple cell types
pb2 = pb[pb.obs["cell_type"].isin([ct1] + ct2_list)].copy()

pb2.obs["cell_type_2lvl"] = pb2.obs["cell_type"].astype(str)
pb2.obs.loc[pb2.obs["cell_type_2lvl"].isin(ct2_list), "cell_type_2lvl"] = "NeuronCombo"
pb2.obs.loc[pb2.obs["cell_type_2lvl"] == ct1, "cell_type_2lvl"] = "AstroProtoplasmic"
pb2.obs["cell_type_2lvl"] = pb2.obs["cell_type_2lvl"].astype("category")

In [ ]:
# of one cell type subset pseudobulk
pb2 = pb[pb.obs["cell_type"].isin([ct1, ct2])].copy()

In [ ]:
# make two dfs necessary for PyDESeq2
counts_df = pd.DataFrame(
    pb2.X.toarray() if sp.issparse(pb2.X) else np.asarray(pb2.X),
    index=pb2.obs_names,
    columns=pb2.var_names,
).astype(int)

metadata_df = pb2.obs[["animal_id", "cell_type"]].copy()

In [ ]:
counts_df

In [ ]:
dds = DeseqDataSet(
    counts=counts_df,
    metadata=metadata_df,
    design_factors=["animal_id", "cell_type"],
    n_cpus=8,         
    refit_cooks=True,
)
dds.deseq2()

In [ ]:
# extracting the contrast
stat_res = DeseqStats(dds, contrast=("cell_type", "Astrocytes Protoplasmic", "CA2/CA3 Neurons"), n_cpus=8)
stat_res.summary()

In [ ]:
res = stat_res.results_df.sort_values("log2FoldChange", ascending=False)
res.head(60)

In [ ]:
df = res.copy()
df["neglog10_padj"] = -np.log10(df["padj"].clip(lower=1e-300))

In [ ]:
import matplotlib.pyplot as plt

alpha = 0.05      # FDR cutoff
lfc_thr = 1.0     # log2FC threshold (optional)

sig = (df["padj"] < alpha) & (df["log2FoldChange"].abs() >= lfc_thr)

plt.figure(figsize=(6, 5))
plt.scatter(df["log2FoldChange"], df["neglog10_padj"], s=12, alpha=0.6)
plt.scatter(df.loc[sig, "log2FoldChange"], df.loc[sig, "neglog10_padj"], s=12, alpha=0.9)

plt.axhline(-np.log10(alpha), linestyle="--")
plt.axvline(lfc_thr, linestyle="--")
plt.axvline(-lfc_thr, linestyle="--")

plt.xlabel("log2 fold change")
plt.ylabel("-log10(adjusted p-value)")
plt.title(f"Volcano: {ct1} vs {ct2}")
plt.tight_layout()
plt.show()

In [ ]:
lfc_thr = 1.0
alpha = 0.1

df = res.copy()

highlight_genes = df.index[
    (df["padj"] < alpha) &
    (df["log2FoldChange"].abs() >= lfc_thr)
].tolist()

In [ ]:
df = res.copy()
df["neglog10_padj"] = -np.log10(df["padj"].clip(lower=1e-300))

# keep only genes that actually exist in the results
highlight_present = [g for g in highlight_genes if g in df.index]
missing = sorted(set(highlight_genes) - set(highlight_present))
if missing:
    print("Not found in results:", missing)

plt.figure(figsize=(20, 10))

# all points
plt.scatter(df["log2FoldChange"], df["neglog10_padj"], s=12, alpha=0.5)

# highlighted points
plt.scatter(
    df.loc[highlight_present, "log2FoldChange"],
    df.loc[highlight_present, "neglog10_padj"],
    s=40,
    alpha=0.9,
)

# label highlighted genes
for g in highlight_present:
    plt.text(
        df.loc[g, "log2FoldChange"],
        df.loc[g, "neglog10_padj"],
        g,
        fontsize=9,
        ha="left",
        va="bottom",
    )

# optional threshold lines
alpha = 0.05
plt.axhline(-np.log10(alpha), linestyle="--")
plt.axvline(1.0, linestyle="--")
plt.axvline(-1.0, linestyle="--")

plt.xlabel("log2 fold change")
plt.ylabel("-log10(adjusted p-value)")
plt.title(f"Volcano: {ct1} vs {ct2}")
plt.tight_layout()
plt.show()

### Making a heatmap

Based on log2FC ranking top10 either way & plotting the normalized log2 expression

In [ ]:
exclude_genes = [
     "Rbfox3",
     "Tubb3",
     "Snap25",
     "Syt1",
     "Slc17a7",
     "Slc17a6",
     "Gad1",
     "Gad2",
     "Aldoc",
     "Aqp4",
     "Aldh1l1",
     "Slc1a3",
     "Slc1a2",
     "Gfap",
     "Frzb",
     "Ascl1",
     "Agt",
     "Fam107a",
     "Plp1",
     "Mog",
     "Csf1r",
     "C1qa",
     "Flt1",
     "Pecam1",
]

In [ ]:
alpha = 0.05
lfc_thr = 1.0
top_n_astro = 10
top_n_other = 5

sig = (
    res.loc[(res["padj"] < alpha) & (res["log2FoldChange"].abs() >= lfc_thr)]
    .drop(index=exclude_genes, errors="ignore")
    .copy()
)

# top in ct1 (positive LFC)
top_ct1 = (
    sig.loc[sig["log2FoldChange"] > 0]
    .sort_values("log2FoldChange", ascending=False)
    .head(top_n_astro)
    .index
    .tolist()
)

# top in ct2 (negative LFC)
top_ct2 = (
    sig.loc[sig["log2FoldChange"] < 0]
    .sort_values("log2FoldChange", ascending=True)
    .head(top_n_other)
    .index
    .tolist()
)

# combined, keeping ct1 first then ct2
candidates = top_ct1 + top_ct2
candidates

In [ ]:
dds.vst()

vst_mat = dds.layers["vst_counts"]

vst_df = pd.DataFrame(
    vst_mat.toarray() if sp.issparse(vst_mat) else np.asarray(vst_mat),
    index=dds.obs_names,
    columns=dds.var_names,
)

In [ ]:
group_col = "cell_type_2lvl" if "cell_type_2lvl" in metadata_df.columns else "cell_type"

meta = metadata_df.copy()
meta[group_col] = meta[group_col].astype(str)
meta["animal_id"] = meta["animal_id"].astype(str)

# sample_id optional, but nice for consistent ordering if present in pb2.obs
if "sample_id" in pb2.obs.columns:
    meta["sample_id"] = pb2.obs.loc[meta.index, "sample_id"].astype(str)
    sort_cols = [group_col, "animal_id", "sample_id"]
else:
    sort_cols = [group_col, "animal_id"]

ordered_samples = meta.sort_values(sort_cols).index.tolist()

In [ ]:
heat = vst_df.loc[ordered_samples, candidates]   # samples x genes
heat = heat.apply(zscore, axis=0)                # z-score per gene
heat = heat.T     

In [ ]:
g = sns.clustermap(
    heat,
    cmap=sns.diverging_palette(145, 300, s=60, as_cmap=True),
    center=0,
    col_cluster=True,   # <-- cluster samples
    row_cluster=True,   # set False if you do NOT want to cluster genes
    xticklabels=False,
    yticklabels=True,
    figsize=(4, 6),
)

plt.show()

In [ ]:
dds.vst()

In [ ]:
dds

In [ ]:
vst_mat = dds.layers["vst_counts"]

In [ ]:
vst_df = pd.DataFrame(
    vst_mat.toarray() if sp.issparse(vst_mat) else np.asarray(vst_mat),
    index=pb2.obs_names,
    columns=pb2.var_names,
)

In [ ]:
genes_use = res.index[
    (res["padj"] < 0.05) &
    (res["log2FoldChange"].abs() >= 1)
].tolist()

In [ ]:
genes_use = (
    res.loc[genes_use]
    .sort_values("padj")
    .head(30)
    .index
    .tolist()
)

In [ ]:
from scipy.stats import zscore

heat_df = vst_df[genes_use]
heat_df = heat_df.apply(zscore, axis=0)

In [ ]:
col_annot = pb2.obs.loc[heat_df.index, ["cell_type", "animal_id"]]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.clustermap(
    heat_df.T,
    col_colors=[
        col_annot["cell_type"].map({
            "Astrocytes Protoplasmic": "#d95f02",
            "CA1 Neurons": "#1b9e77",
        }),
        col_annot["animal_id"].map({
            "animal_1": "#7570b3",
            "animal_2": "#e7298a",
            "animal_3": "#66a61e",
        }),
    ],
    cmap="vlag",
    center=0,
    figsize=(10, 8),
    xticklabels=False,
)
plt.show()